# Analysis workflow tests

Loads the survey and sensor data, builds a panel dataset, and converts it for two kinds of analysis:

- [SurvivalScale](https://github.com/tazzben/Likert-estimation) (section 5), for Likert-scale responses as the dependent variable.
- OLS with [statsmodels](https://www.statsmodels.org/) (section 6), for a continuous dependent variable such as a slider response or a sensor measure.

Each section ends with assertions that fail loudly if something is wrong.

Install the estimators with `pip install SurvivalScale statsmodels` to run the final sections.

In [18]:
import sqlite3
import pandas as pd
from pathlib import Path
import sys

for project_root in (Path.cwd(), *Path.cwd().parents):
    src_dir = project_root / "src"
    if (src_dir / "CBACABLab").is_dir():
        sys.path.insert(0, str(src_dir))
        break

import CBACABLab

questionIdToQuestionName = {
    "QID1": "Q1",
    "QID2": "Q2",
    "QID3": "Q3",
    "QID4": "Q4",
    "QID7": "Q5",
    "QID5": "Q6",
    "QID6": "Q7"
}

DATABASE = "data.db"
SURVEY_FILE = "Survey.xlsx"
SENSOR_FOLDER = "./sensorData/"
REBUILD = True  # reload the survey and sensor data; set False to reuse the existing database

# Respondent-level slider and demographic questions
RESPONDENT_QUESTIONS = ["Q4_4", "Q5_4", "Q6", "Q7"]

# Questions with a Likert-style response, and the scale for each.
# This five-point scale matches the labels used by Q1-Q3 in Survey.xlsx.
AGREEMENT_SCALE = {
    "Strongly disagree": 0,
    "Somewhat disagree": 1,
    "Neither agree nor disagree": 2,
    "Somewhat agree": 3,
    "Strongly agree": 4,
}
LIKERT_SCALES = {question: AGREEMENT_SCALE for question in ["Q1", "Q2", "Q3"]}

TAGS = ["left", "right"]
INSTRUMENT_PROPERTIES = ["Attention", "Joy"]

## 1. Load the survey and sensor data

In [19]:
if REBUILD:
    CBACABLab.LoadSurveyData(databaseName=DATABASE, surveyName=SURVEY_FILE)
    CBACABLab.LoadSensorData(
        databaseName=DATABASE,
        sensorFolder=SENSOR_FOLDER,
        boundedboxes=[(0, 960, 0, 1080)],
        tag="left",
        rebuild=True,
        questionIdtoQuestionName=questionIdToQuestionName
    )
    CBACABLab.LoadSensorData(
        databaseName=DATABASE,
        sensorFolder=SENSOR_FOLDER,
        boundedboxes=[(960, 1920, 0, 1080)],
        tag="right",
        questionIdtoQuestionName=questionIdToQuestionName
    )


In [20]:
# Test: the tables exist and the survey has data
with sqlite3.connect(DATABASE) as con:
    counts = {
        table: con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        for table in ["FullSurvey", "SensorData", "InstrumentData"]
    }
print(counts)
assert counts["FullSurvey"] > 0, "FullSurvey is empty"
if counts["SensorData"] == 0:
    print("WARNING: SensorData is empty. Sensor columns will be NaN; check the sensor folder and respondent ids.")

{'FullSurvey': 100, 'SensorData': 140, 'InstrumentData': 4993}


In [21]:
# Test: every question we plan to use is in the survey
available = set(CBACABLab.ListQuestions(DATABASE))
wanted = set(RESPONDENT_QUESTIONS) | set(LIKERT_SCALES)
missing = wanted - available
assert not missing, f"Questions not found in the survey: {sorted(missing)}"

## 2. Build the panel dataset

In [22]:
panel = CBACABLab.CreatePanelData(
    question_level_columns=list(LIKERT_SCALES),
    respondent_level_columns=RESPONDENT_QUESTIONS,
    tagFilter=TAGS,
    instrumentProperties=INSTRUMENT_PROPERTIES,
    databaseName=DATABASE,
)
panel.head(10)

,respondent_id,question,response,duration_left,duration_right,pupil_left,pupil_right,Attention_left,Attention_right,Joy_left,Joy_right,Q4_4,Q5_4,Q6,Q7
0,1,Q1,Strongly disagree,343.472949,357.075008,2.628579,2.714873,98.137547,97.839279,0.034117,0.037362,65.198,22.066,Other,Bachelor's Degree
1,1,Q2,Neither agree nor disagree,358.868249,1916.269888,2.776471,2.833336,97.348034,98.041817,0.120167,1.596050,65.198,22.066,Other,Bachelor's Degree
2,1,Q3,Strongly agree,409.204696,694.727733,2.817981,2.836787,NaN,97.550504,NaN,0.052147,65.198,22.066,Other,Bachelor's Degree
3,2,Q1,Strongly agree,384.045313,722.940208,2.693345,2.841391,97.755627,97.696284,0.025384,0.025649,89.207,94.341,White/ Caucasian,Prefer not to say
4,2,Q2,Neither agree nor disagree,1027.143104,398.548022,2.739849,2.741312,95.377090,97.866264,0.025275,0.024139,89.207,94.341,White/ Caucasian,Prefer not to say
5,2,Q3,Somewhat disagree,736.664201,NaN,2.752320,NaN,97.902746,NaN,25.002819,NaN,89.207,94.341,White/ Caucasian,Prefer not to say
6,3,Q1,Neither agree nor disagree,380.960150,264.400428,2.719926,2.847198,98.039979,98.498688,26.620419,0.036503,100.000,2.458,White/ Caucasian,High school diploma / GED
7,3,Q2,Strongly disagree,591.957592,438.112677,2.702798,2.532557,97.798824,98.661983,90.557137,0.024104,100.000,2.458,White/ Caucasian,High school diploma / GED
8,3,Q3,Somewhat disagree,733.998717,257.911149,2.662162,2.658208,97.131732,98.370094,0.029900,0.038205,100.000,2.458,White/ Caucasian,High school diploma / GED
9,4,Q1,Somewhat agree,385.729328,402.003738,2.624093,2.658857,97.832081,NaN,0.043469,NaN,51.432,7.629,Asian/ Asian American,Master's Degree


In [23]:
# Tests: panel structure
required = {"respondent_id", "question", "response", *RESPONDENT_QUESTIONS}
assert required <= set(panel.columns), f"Missing columns: {required - set(panel.columns)}"
assert set(panel["question"]) == set(LIKERT_SCALES), "Panel contains unexpected questions"
sensor_columns = [c for c in panel.columns if c.startswith(("pupil_", "duration_"))]
if counts["SensorData"] > 0:
    assert sensor_columns and panel[sensor_columns].notna().any().any(), "No sensor data matched the survey questions"

# Strictly one row per respondent, question, tag
assert not panel.duplicated(["respondent_id", "question"]).any(), "More than one row per respondent and question"

## 3. Convert responses to `k` and `bound`

Responses outside a question's scale (for example "Don't know") can't be converted, so they are listed and dropped first.

In [24]:
def labels(scale):
    if isinstance(scale, str):
        scale = dict(CBACABLab.ListLikertScales(show_values=True))[scale]
    return {str(label).strip().lower() for label in scale}

in_scale = pd.Series(
    [str(r).strip().lower() in labels(LIKERT_SCALES[q]) for q, r in zip(panel["question"], panel["response"])],
    index=panel.index,
)
print("Dropping responses outside their scale:")
print(panel.loc[~in_scale, ["question", "response"]].value_counts())
panel_likert = CBACABLab.AddLikertColumns(panel[in_scale], LIKERT_SCALES)
panel_likert.head(10)

Dropping responses outside their scale:
Series([], Name: count, dtype: int64)


,respondent_id,question,response,duration_left,duration_right,pupil_left,pupil_right,Attention_left,Attention_right,Joy_left,Joy_right,Q4_4,Q5_4,Q6,Q7,k,bound
0,1,Q1,Strongly disagree,343.472949,357.075008,2.628579,2.714873,98.137547,97.839279,0.034117,0.037362,65.198,22.066,Other,Bachelor's Degree,0,4
1,1,Q2,Neither agree nor disagree,358.868249,1916.269888,2.776471,2.833336,97.348034,98.041817,0.120167,1.596050,65.198,22.066,Other,Bachelor's Degree,2,4
2,1,Q3,Strongly agree,409.204696,694.727733,2.817981,2.836787,NaN,97.550504,NaN,0.052147,65.198,22.066,Other,Bachelor's Degree,4,4
3,2,Q1,Strongly agree,384.045313,722.940208,2.693345,2.841391,97.755627,97.696284,0.025384,0.025649,89.207,94.341,White/ Caucasian,Prefer not to say,4,4
4,2,Q2,Neither agree nor disagree,1027.143104,398.548022,2.739849,2.741312,95.377090,97.866264,0.025275,0.024139,89.207,94.341,White/ Caucasian,Prefer not to say,2,4
5,2,Q3,Somewhat disagree,736.664201,NaN,2.752320,NaN,97.902746,NaN,25.002819,NaN,89.207,94.341,White/ Caucasian,Prefer not to say,1,4
6,3,Q1,Neither agree nor disagree,380.960150,264.400428,2.719926,2.847198,98.039979,98.498688,26.620419,0.036503,100.000,2.458,White/ Caucasian,High school diploma / GED,2,4
7,3,Q2,Strongly disagree,591.957592,438.112677,2.702798,2.532557,97.798824,98.661983,90.557137,0.024104,100.000,2.458,White/ Caucasian,High school diploma / GED,0,4
8,3,Q3,Somewhat disagree,733.998717,257.911149,2.662162,2.658208,97.131732,98.370094,0.029900,0.038205,100.000,2.458,White/ Caucasian,High school diploma / GED,1,4
9,4,Q1,Somewhat agree,385.729328,402.003738,2.624093,2.658857,97.832081,NaN,0.043469,NaN,51.432,7.629,Asian/ Asian American,Master's Degree,3,4


## 4. Prepare the data for SurvivalScale

SurvivalScale expects one row per observation with `id`, `block`, `question`, numeric feature columns, `k` and `bound`. Here `block` is the respondent, so errors can be clustered by respondent.

In [25]:
base = panel_likert.reset_index(drop=True)

FEATURES = ["pupil_left"]
survival = pd.DataFrame({
    "id":  base["respondent_id"],
    "block": base["respondent_id"],
    "question": base["question"],
    "pupil_left": pd.to_numeric(base["pupil_left"], errors="coerce"),
    "k": base["k"].astype(int),
    "bound": base["bound"].astype(int),
}).dropna()
survival.head(10)


,id,block,question,pupil_left,k,bound
0,1,1,Q1,2.628579,0,4
1,1,1,Q2,2.776471,2,4
2,1,1,Q3,2.817981,4,4
3,2,2,Q1,2.693345,4,4
4,2,2,Q2,2.739849,2,4
5,2,2,Q3,2.752320,1,4
6,3,3,Q1,2.719926,2,4
7,3,3,Q2,2.702798,0,4
8,3,3,Q3,2.662162,1,4
9,4,4,Q1,2.624093,3,4


## 5. Run the estimator

In [29]:

# If necessary, install the SurvivalScale package. Uncomment the following line to install it in a Jupyter notebook environment.
# %pip install SurvivalScale 


from SurvivalScale import get_results

if get_results is not None:
    cj, beta, metrics = get_results(
        survival,
        bootstrap_iterations=100,
        alpha=0.05,
        columns=FEATURES,
        block_id="block",
    )
    assert cj is not None and beta is not None and metrics is not None
    display(beta)

Performing bootstrap analysis to estimate confidence intervals and p-values...


100%|██████████| 100/100 [00:02<00:00, 41.61it/s]


,beta,marginal,marginal_discrete,lower_bound,upper_bound,p_value
pupil_left,-0.392776,-0.075341,-0.096951,-0.571478,-0.226246,0.0


## 6. OLS with statsmodels

OLS needs a continuous outcome. Likert responses are ordinal, so here the continuous slider response `Q4_4` is the dependent variable, with the Likert response `k` and the other slider `Q5_4` as regressors. Question fixed effects control for the three survey items, and standard errors are clustered by respondent because each respondent appears once per question.

In [27]:
# If necessary, install the statsmodels package. Uncomment the following line to install it in a Jupyter notebook environment.
# %pip install statsmodels

import statsmodels.formula.api as smf

ols_data = survival.copy()
ols_data["question"] = ols_data["question"].astype("category")

ols_model = smf.ols("pupil_left ~ C(question)", data=ols_data).fit(
    cov_type="cluster", cov_kwds={"groups": ols_data["block"]}
)
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:             pupil_left   R-squared:                       0.113
Model:                            OLS   Adj. R-squared:                  0.047
Method:                 Least Squares   F-statistic:                     1.619
Date:                Wed, 04 Nov 2026   Prob (F-statistic):              0.251
Time:                        17:48:59   Log-Likelihood:                 38.452
No. Observations:                  30   AIC:                            -70.90
Df Residuals:                      27   BIC:                            -66.70
Df Model:                           2                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             2.6455      0.02